<a href="https://colab.research.google.com/github/AidoruFusion/northstar-databases-analytics/blob/main/MongoDB_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 02: MongoDB Atlas Upload, CRUD, Aggregation and Indexing

This notebook uploads the cleaned NorthStar JSON data into MongoDB Atlas and demonstrates NoSQL database development using PyMongo.

The notebook includes:
- MongoDB Atlas connection using a hidden URI prompt;
- uploading cleaned JSON files into collections;
- NoSQL document design for an integrated operational view;
- CRUD operations;
- aggregation pipelines;
- indexing and query optimisation using `explain`.


## Step 1: Install and import packages

This step imports the Python packages needed for MongoDB work.

- `pymongo` connects Python to MongoDB Atlas.
- `pandas` supports checking and preparing data.
- `json` loads cleaned JSON files.
- `pathlib` manages project folders.

In [1]:
!pip install pymongo dnspython -q

import json
import pandas as pd
from pathlib import Path
from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import ServerSelectionTimeoutError, OperationFailure

## Step 2: Set MongoDB connection details

This notebook connects to MongoDB Atlas using a hidden input prompt.

The MongoDB URI is **not written directly into the notebook** because the notebook will be saved to GitHub and the URI contains a username and password.

When you run the next cell in Google Colab, paste your MongoDB Atlas URI into the hidden prompt.


In [2]:
from getpass import getpass

# Paste your MongoDB Atlas URI when prompted.
# The input is hidden and is not saved in the notebook output.
MONGO_URI = getpass("Paste MongoDB Atlas URI: ")

DATABASE_NAME = "North-Database"

if MONGO_URI.strip() == "":
    print("MongoDB URI is blank. Paste your connection string when prompted before running connection cells.")
else:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    db = client[DATABASE_NAME]
    print("Connected successfully to MongoDB Atlas.")


Paste MongoDB Atlas URI: ··········
Connected successfully to MongoDB Atlas.


## Step 3: Set the JSON data folder

The cleaned JSON files should be saved in the GitHub repository folder:

`data/cleaned/json/`

This keeps the workflow clear:

raw CSV → cleaned CSV → cleaned JSON → MongoDB Atlas.

In [3]:
repo_folder = Path("/content/northstar-databases-analytics")
json_folder = repo_folder / "data/cleaned/json"

print("JSON folder:", json_folder)
print("Folder exists:", json_folder.exists())

if json_folder.exists():
    print("JSON files found:")
    for file in sorted(json_folder.glob("*.json")):
        print("-", file.name)

JSON folder: /content/northstar-databases-analytics/data/cleaned/json
Folder exists: False


## Step 4: Define MongoDB collection names

Each JSON file is mapped to a MongoDB collection. The names are aligned with the existing NorthStar database shown in MongoDB Atlas.

In [4]:
collection_map = {
    "customers.json": "Clean_Customers",
    "orders.json": "Cleaned_Orders",
    "deliveries.json": "Cleaned_Deliveries",
    "drivers.json": "Clean_Driver",
    "vehicles.json": "Cleaned_Vehicles",
    "hubs.json": "Clean_Hubs",
    "complaints.json": "Clean_Complaints",
    "incidents.json": "Cleaned_Incidents",
    "app_events.json": "Clean_app_events",
    "data_dictionary.json": "Cleaned_data_dictionary"
}

collection_map

{'customers.json': 'Clean_Customers',
 'orders.json': 'Cleaned_Orders',
 'deliveries.json': 'Cleaned_Deliveries',
 'drivers.json': 'Clean_Driver',
 'vehicles.json': 'Cleaned_Vehicles',
 'hubs.json': 'Clean_Hubs',
 'complaints.json': 'Clean_Complaints',
 'incidents.json': 'Cleaned_Incidents',
 'app_events.json': 'Clean_app_events',
 'data_dictionary.json': 'Cleaned_data_dictionary'}

## Step 5: Load JSON files into MongoDB Atlas

This step reads each cleaned JSON file and inserts it into the matching MongoDB collection.

For reproducibility, each collection is cleared before inserting the cleaned records. This prevents duplicates if the notebook is run more than once.

In [5]:
def load_json_collection(json_path, collection_name):
    with open(json_path, "r", encoding="utf-8") as file:
        records = json.load(file)

    collection = db[collection_name]
    collection.delete_many({})

    if len(records) > 0:
        collection.insert_many(records)

    print(f"{collection_name}: inserted {len(records)} records")


if MONGO_URI != "":
    for json_file, collection_name in collection_map.items():
        json_path = json_folder / json_file

        if json_path.exists():
            load_json_collection(json_path, collection_name)
        else:
            print(f"Missing file: {json_file}")
else:
    print("Skipping upload because MONGO_URI is blank.")

Missing file: customers.json
Missing file: orders.json
Missing file: deliveries.json
Missing file: drivers.json
Missing file: vehicles.json
Missing file: hubs.json
Missing file: complaints.json
Missing file: incidents.json
Missing file: app_events.json
Missing file: data_dictionary.json


## Step 6: Verify uploaded collections

This confirms that the records were uploaded successfully.

In [6]:
if MONGO_URI != "":
    for collection_name in collection_map.values():
        count = db[collection_name].count_documents({})
        print(f"{collection_name}: {count} documents")
else:
    print("Skipping verification because MONGO_URI is blank.")

Clean_Customers: 650 documents
Cleaned_Orders: 1250 documents
Cleaned_Deliveries: 950 documents
Clean_Driver: 170 documents
Cleaned_Vehicles: 120 documents
Clean_Hubs: 8 documents
Clean_Complaints: 320 documents
Cleaned_Incidents: 280 documents
Clean_app_events: 640 documents
Cleaned_data_dictionary: 9 documents


# NoSQL Design Rationale

NorthStar has both structured and semi-structured data.

This notebook keeps cleaned source files as separate collections, but also creates an integrated case-level collection. This supports two types of analysis:

- separate collections for direct operational querying;
- integrated documents for viewing customer, order, delivery, complaint, incident, and app-event information together.

This reflects the case study problem, where NorthStar needs to connect service failures, complaints, deliveries, incidents, and customer activity.

## Step 7: Create an integrated operational case collection

This creates a document-based view around each order.

Each document can contain order details, customer summary, delivery summary, related complaints, related incidents, and related app events.

In [7]:
def create_operational_cases():
    cases = []
    orders = list(db["Cleaned_Orders"].find({}, {"_id": 0}))

    for order in orders:
        order_id = order.get("order_id")
        customer_id = order.get("customer_id")

        customer = db["Clean_Customers"].find_one({"customer_id": customer_id}, {"_id": 0})
        delivery = db["Cleaned_Deliveries"].find_one({"order_id": order_id}, {"_id": 0})
        complaints = list(db["Clean_Complaints"].find({"order_id": order_id}, {"_id": 0}))

        incidents = []
        if delivery and delivery.get("delivery_id"):
            incidents = list(db["Cleaned_Incidents"].find({"delivery_id": delivery.get("delivery_id")}, {"_id": 0}))

        app_events = list(db["Clean_app_events"].find({"order_id": order_id}, {"_id": 0}))

        case_document = {
            "case_id": f"CASE_{order_id}",
            "order_id": order_id,
            "customer_id": customer_id,
            "customer": customer,
            "order": order,
            "delivery": delivery,
            "complaints": complaints,
            "incidents": incidents,
            "app_events": app_events,
            "complaint_count": len(complaints),
            "incident_count": len(incidents),
            "app_event_count": len(app_events)
        }

        cases.append(case_document)

    db["Operational_Cases"].delete_many({})

    if cases:
        db["Operational_Cases"].insert_many(cases)

    print(f"Operational_Cases inserted: {len(cases)}")


if MONGO_URI != "":
    create_operational_cases()
else:
    print("Skipping integrated collection creation because MONGO_URI is blank.")

Operational_Cases inserted: 1250


## Step 8: Preview one integrated operational case

This checks whether the nested document structure has been created correctly.

In [8]:
if MONGO_URI != "":
    sample_case = db["Operational_Cases"].find_one({}, {"_id": 0})
    sample_case
else:
    print("Skipping preview because MONGO_URI is blank.")

# CRUD Operations

This section demonstrates MongoDB CRUD operations using PyMongo.

CRUD stands for Create, Read, Update, and Delete.

## Step 9: CREATE operation

This inserts a test service review document. It is marked as a demo record so it can be removed later.

In [9]:
if MONGO_URI != "":
    demo_record = {
        "case_id": "DEMO_CASE_001",
        "order_id": "DEMO_ORDER_001",
        "customer_id": "DEMO_CUSTOMER_001",
        "issue_type": "Late Arrival",
        "zone": "Central",
        "status": "Open",
        "priority": "High",
        "notes": "Demo CRUD record created for coursework testing."
    }

    result = db["Service_Review_Demo"].insert_one(demo_record)
    print("Inserted demo document ID:", result.inserted_id)
else:
    print("Skipping create operation because MONGO_URI is blank.")

Inserted demo document ID: 69f683f55dc12de8f32fe0e5


## Step 10: READ operations

These queries demonstrate simple filtering, projection, sorting, and limiting results.

In [10]:
if MONGO_URI != "":
    print("Customers in Central zone:")
    for doc in db["Clean_Customers"].find(
        {"home_zone": "Central"},
        {"_id": 0, "customer_id": 1, "home_zone": 1, "customer_type": 1, "loyalty_score": 1}
    ).limit(5):
        print(doc)

    print("\nHigh-priority orders:")
    for doc in db["Cleaned_Orders"].find(
        {"priority_level": {"$in": ["High", "Urgent"]}},
        {"_id": 0, "order_id": 1, "customer_id": 1, "pickup_zone": 1, "priority_level": 1, "order_value": 1}
    ).sort("order_value", DESCENDING).limit(5):
        print(doc)
else:
    print("Skipping read operations because MONGO_URI is blank.")

Customers in Central zone:
{'customer_id': 'C0058', 'home_zone': 'Central', 'customer_type': 'SME', 'loyalty_score': None}
{'customer_id': 'C0108', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 96}
{'customer_id': 'C0028', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 78.5}
{'customer_id': 'C0053', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 45.7}
{'customer_id': 'C0201', 'home_zone': 'Central', 'customer_type': 'Consumer', 'loyalty_score': 69.2}

High-priority orders:
{'order_id': 'O00144', 'customer_id': 'C0063', 'pickup_zone': 'Riverside', 'priority_level': 'High', 'order_value': 288.86}
{'order_id': 'O00790', 'customer_id': 'C0028', 'pickup_zone': 'Airport', 'priority_level': 'High', 'order_value': 283.08}
{'order_id': 'O00799', 'customer_id': 'C0289', 'pickup_zone': 'East', 'priority_level': 'High', 'order_value': 260}
{'order_id': 'O00115', 'customer_id': 'C0157', 'pickup_zone': 'East', 'priority_level': 'H

## Step 11: UPDATE operation

This updates the demo service review record.

In [11]:
if MONGO_URI != "":
    result = db["Service_Review_Demo"].update_one(
        {"case_id": "DEMO_CASE_001"},
        {"$set": {"status": "Investigating", "assigned_team": "Operations"}}
    )

    print("Documents matched:", result.matched_count)
    print("Documents modified:", result.modified_count)

    updated_doc = db["Service_Review_Demo"].find_one({"case_id": "DEMO_CASE_001"}, {"_id": 0})
    print(updated_doc)
else:
    print("Skipping update operation because MONGO_URI is blank.")

Documents matched: 1
Documents modified: 1
{'case_id': 'DEMO_CASE_001', 'order_id': 'DEMO_ORDER_001', 'customer_id': 'DEMO_CUSTOMER_001', 'issue_type': 'Late Arrival', 'zone': 'Central', 'status': 'Investigating', 'priority': 'High', 'notes': 'Demo CRUD record created for coursework testing.', 'assigned_team': 'Operations'}


## Step 12: DELETE operation

This removes the demo record after testing.

In [12]:
if MONGO_URI != "":
    result = db["Service_Review_Demo"].delete_one({"case_id": "DEMO_CASE_001"})
    print("Documents deleted:", result.deleted_count)
else:
    print("Skipping delete operation because MONGO_URI is blank.")

Documents deleted: 1


# Aggregation Pipelines

MongoDB aggregation pipelines process documents in stages. This section uses aggregation to answer NorthStar-style operational questions.

## Step 13: Aggregation — order value by service type and zone

This identifies which service types and zones generate the highest order value.

In [13]:
if MONGO_URI != "":
    pipeline = [
        {"$group": {
            "_id": {"service_type": "$service_type", "pickup_zone": "$pickup_zone"},
            "total_order_value": {"$sum": "$order_value"},
            "average_order_value": {"$avg": "$order_value"},
            "order_count": {"$sum": 1}
        }},
        {"$sort": {"total_order_value": -1}},
        {"$limit": 10}
    ]

    results = list(db["Cleaned_Orders"].aggregate(pipeline))
    display(pd.DataFrame(results))
else:
    print("Skipping aggregation because MONGO_URI is blank.")

,_id,total_order_value,average_order_value,order_count
0,"{'service_type': 'Passenger', 'pickup_zone': '...",5892.65,107.139091,55
1,"{'service_type': 'Passenger', 'pickup_zone': '...",5678.06,84.747164,67
2,"{'service_type': 'Retail', 'pickup_zone': 'Cen...",5441.86,83.720923,65
3,"{'service_type': 'Passenger', 'pickup_zone': '...",4891.96,94.076154,52
4,"{'service_type': 'Parcel', 'pickup_zone': 'East'}",4833.41,92.950192,52
5,"{'service_type': 'Parcel', 'pickup_zone': 'Cen...",4647.96,84.508364,55
6,"{'service_type': 'Passenger', 'pickup_zone': '...",4467.14,95.045532,47
7,"{'service_type': 'Passenger', 'pickup_zone': '...",4244.66,108.837436,39
8,"{'service_type': 'Passenger', 'pickup_zone': '...",4189.70,99.754762,42
9,"{'service_type': 'Retail', 'pickup_zone': 'Sou...",4133.59,98.418810,42


## Step 14: Aggregation — delivery performance by hub

This supports the case study issue of underperforming hubs and service delays.

In [14]:
if MONGO_URI != "":
    pipeline = [
        {"$group": {
            "_id": "$hub_id",
            "delivery_count": {"$sum": 1},
            "avg_duration_hours": {"$avg": "$delivery_duration_hours"},
            "avg_rating": {"$avg": "$customer_rating_post_delivery"},
            "avg_cost": {"$avg": "$fuel_or_charge_cost"},
            "manual_override_total": {"$sum": "$manual_route_override_count"}
        }},
        {"$sort": {"avg_duration_hours": -1}}
    ]

    results = list(db["Cleaned_Deliveries"].aggregate(pipeline))
    display(pd.DataFrame(results).head(10))
else:
    print("Skipping aggregation because MONGO_URI is blank.")

,_id,delivery_count,avg_duration_hours,avg_rating,avg_cost,manual_override_total
0,H05,115,11.553300,3.669558,13.686000,109
1,H04,127,11.102635,3.915476,13.167008,111
2,H01,136,10.684968,3.840593,12.755809,140
3,H08,128,10.560490,3.884560,11.708203,142
4,H07,115,10.535888,3.881858,12.922087,121
5,H06,104,9.917423,3.882136,13.319231,95
6,H02,106,9.478948,3.950952,12.565000,97
7,H03,119,8.437893,3.895862,12.744202,106


## Step 15: Aggregation — complaints by type and severity

This helps identify the main causes of customer dissatisfaction.

In [15]:
if MONGO_URI != "":
    pipeline = [
        {"$group": {
            "_id": {"complaint_type": "$complaint_type", "severity": "$severity"},
            "complaint_count": {"$sum": 1},
            "avg_resolution_days": {"$avg": "$resolution_days"},
            "total_compensation": {"$sum": "$compensation_amount"}
        }},
        {"$sort": {"complaint_count": -1}}
    ]

    results = list(db["Clean_Complaints"].aggregate(pipeline))
    display(pd.DataFrame(results).head(10))
else:
    print("Skipping aggregation because MONGO_URI is blank.")

,_id,complaint_count,avg_resolution_days,total_compensation
0,"{'complaint_type': 'Delay', 'severity': 'Medium'}",56,5.964286,964.87
1,"{'complaint_type': 'MissedPickup', 'severity':...",37,6.162162,644.80
2,"{'complaint_type': 'DriverBehaviour', 'severit...",31,5.419355,476.29
3,"{'complaint_type': 'Delay', 'severity': 'Low'}",27,6.481481,220.43
4,"{'complaint_type': 'AppIssue', 'severity': 'Me...",25,7.360000,386.58
5,"{'complaint_type': 'Delay', 'severity': 'High'}",18,12.444444,511.54
6,"{'complaint_type': 'MissedPickup', 'severity':...",16,11.562500,689.11
7,"{'complaint_type': 'DriverBehaviour', 'severit...",16,13.750000,460.63
8,"{'complaint_type': 'AppIssue', 'severity': 'Low'}",15,6.066667,185.55
9,"{'complaint_type': 'AppIssue', 'severity': 'Hi...",13,13.923077,408.59


## Step 16: Aggregation with `$lookup` — deliveries joined to hubs

This joins deliveries to hub information to compare operational performance by hub zone.

In [16]:
if MONGO_URI != "":
    pipeline = [
        {"$lookup": {
            "from": "Clean_Hubs",
            "localField": "hub_id",
            "foreignField": "hub_id",
            "as": "hub_info"
        }},
        {"$unwind": "$hub_info"},
        {"$group": {
            "_id": "$hub_info.zone",
            "delivery_count": {"$sum": 1},
            "avg_delivery_duration": {"$avg": "$delivery_duration_hours"},
            "avg_customer_rating": {"$avg": "$customer_rating_post_delivery"},
            "avg_cost": {"$avg": "$fuel_or_charge_cost"}
        }},
        {"$sort": {"avg_delivery_duration": -1}}
    ]

    results = list(db["Cleaned_Deliveries"].aggregate(pipeline))
    display(pd.DataFrame(results))
else:
    print("Skipping lookup aggregation because MONGO_URI is blank.")

,_id,delivery_count,avg_delivery_duration,avg_customer_rating,avg_cost
0,West,127,11.102635,3.915476,13.167008
1,Central,243,11.034930,3.782479,12.644198
2,North,136,10.684968,3.840593,12.755809
3,Riverside,115,10.535888,3.881858,12.922087
4,Airport,104,9.917423,3.882136,13.319231
5,South,106,9.478948,3.950952,12.565000
6,East,119,8.437893,3.895862,12.744202


## Step 17: Aggregation — operational cases with complaints or incidents

This uses the integrated document collection to identify orders with service problems.

In [17]:
if MONGO_URI != "":
    pipeline = [
        {"$match": {"$or": [{"complaint_count": {"$gt": 0}}, {"incident_count": {"$gt": 0}}]}},
        {"$project": {
            "_id": 0,
            "case_id": 1,
            "order_id": 1,
            "customer_id": 1,
            "complaint_count": 1,
            "incident_count": 1,
            "app_event_count": 1,
            "pickup_zone": "$order.pickup_zone",
            "service_type": "$order.service_type",
            "delivery_status": "$delivery.delivery_status",
            "delivery_duration_hours": "$delivery.delivery_duration_hours"
        }},
        {"$sort": {"complaint_count": -1, "incident_count": -1}},
        {"$limit": 10}
    ]

    results = list(db["Operational_Cases"].aggregate(pipeline))
    display(pd.DataFrame(results))
else:
    print("Skipping operational case aggregation because MONGO_URI is blank.")

,case_id,order_id,customer_id,complaint_count,incident_count,app_event_count,pickup_zone,service_type,delivery_status,delivery_duration_hours
0,CASE_O00125,O00125,C0191,3,0,0,West,Retail,NaN,NaN
1,CASE_O00795,O00795,C0142,3,0,0,Riverside,Retail,NaN,NaN
2,CASE_O01151,O01151,C0100,2,2,0,South,Retail,OnTime,12.478753
3,CASE_O00483,O00483,C0285,2,1,0,Riverside,Business,Failed,14.982770
4,CASE_O01104,O01104,C0480,2,1,0,East,Parcel,OnTime,11.079243
5,CASE_O00351,O00351,C0259,2,1,0,Airport,Parcel,OnTime,0.640782
6,CASE_O00628,O00628,C0056,2,1,1,Riverside,Passenger,Delayed,15.098799
7,CASE_O00417,O00417,C0486,2,1,0,South,Retail,OnTime,3.717208
8,CASE_O00443,O00443,C0378,2,1,0,North,Parcel,OnTime,8.984128
9,CASE_O00167,O00167,C0325,2,0,1,East,Medical,OnTime,22.545555


# Indexing and Query Optimisation

Indexes improve query performance by reducing the number of documents MongoDB must scan.

## Step 18: Create indexes

The indexes support customer lookups, order lookups, delivery status analysis, complaint analysis, app event analysis, and operational case analysis.

In [18]:
if MONGO_URI != "":
    db["Clean_Customers"].create_index([("customer_id", ASCENDING)], name="idx_customer_id")
    db["Clean_Customers"].create_index([("home_zone", ASCENDING)], name="idx_home_zone")

    db["Cleaned_Orders"].create_index([("order_id", ASCENDING)], name="idx_order_id")
    db["Cleaned_Orders"].create_index([("customer_id", ASCENDING)], name="idx_order_customer_id")
    db["Cleaned_Orders"].create_index([("pickup_zone", ASCENDING), ("service_type", ASCENDING)], name="idx_zone_service")

    db["Cleaned_Deliveries"].create_index([("delivery_id", ASCENDING)], name="idx_delivery_id")
    db["Cleaned_Deliveries"].create_index([("order_id", ASCENDING)], name="idx_delivery_order_id")
    db["Cleaned_Deliveries"].create_index([("hub_id", ASCENDING), ("delivery_status", ASCENDING)], name="idx_hub_status")

    db["Clean_Complaints"].create_index([("order_id", ASCENDING)], name="idx_complaint_order_id")
    db["Clean_Complaints"].create_index([("complaint_type", ASCENDING), ("severity", ASCENDING)], name="idx_type_severity")

    db["Cleaned_Incidents"].create_index([("delivery_id", ASCENDING)], name="idx_incident_delivery_id")
    db["Clean_app_events"].create_index([("order_id", ASCENDING)], name="idx_event_order_id")
    db["Clean_app_events"].create_index([("zone_context", ASCENDING)], name="idx_event_zone")

    db["Operational_Cases"].create_index([("order_id", ASCENDING)], name="idx_case_order_id")
    db["Operational_Cases"].create_index([("customer_id", ASCENDING)], name="idx_case_customer_id")
    db["Operational_Cases"].create_index([("complaint_count", DESCENDING), ("incident_count", DESCENDING)], name="idx_case_problem_counts")

    print("Indexes created successfully.")
else:
    print("Skipping index creation because MONGO_URI is blank.")

Indexes created successfully.


## Step 19: Display indexes

This confirms that the indexes have been created.

In [19]:
if MONGO_URI != "":
    for collection_name in ["Clean_Customers", "Cleaned_Orders", "Cleaned_Deliveries", "Clean_Complaints", "Operational_Cases"]:
        print("\nIndexes for:", collection_name)
        for index in db[collection_name].list_indexes():
            print(index)
else:
    print("Skipping index display because MONGO_URI is blank.")


Indexes for: Clean_Customers
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'idx_customer_id')])
SON([('v', 2), ('key', SON([('home_zone', 1)])), ('name', 'idx_home_zone')])

Indexes for: Cleaned_Orders
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'idx_order_id')])
SON([('v', 2), ('key', SON([('customer_id', 1)])), ('name', 'idx_order_customer_id')])
SON([('v', 2), ('key', SON([('pickup_zone', 1), ('service_type', 1)])), ('name', 'idx_zone_service')])

Indexes for: Cleaned_Deliveries
SON([('v', 2), ('key', SON([('_id', 1)])), ('name', '_id_')])
SON([('v', 2), ('key', SON([('delivery_id', 1)])), ('name', 'idx_delivery_id')])
SON([('v', 2), ('key', SON([('order_id', 1)])), ('name', 'idx_delivery_order_id')])
SON([('v', 2), ('key', SON([('hub_id', 1), ('delivery_status', 1)])), ('name', 'idx_hub_status')])

Indexes for: Clean_Complaints
SON(

## Step 20: Use `explain` to check query plan

This demonstrates query optimisation by showing whether MongoDB uses an index.

In [20]:
def explain_find(collection_name, filter_query):
    explain_result = db.command(
        "explain",
        {"find": collection_name, "filter": filter_query},
        verbosity="executionStats"
    )

    execution_stats = explain_result.get("executionStats", {})
    query_planner = explain_result.get("queryPlanner", {})

    print("Collection:", collection_name)
    print("Filter:", filter_query)
    print("Documents returned:", execution_stats.get("nReturned"))
    print("Documents examined:", execution_stats.get("totalDocsExamined"))
    print("Execution time ms:", execution_stats.get("executionTimeMillis"))
    print("Winning plan:")
    print(query_planner.get("winningPlan"))


if MONGO_URI != "":
    explain_find("Cleaned_Orders", {"pickup_zone": "Central", "service_type": "Parcel"})
else:
    print("Skipping explain plan because MONGO_URI is blank.")

Collection: Cleaned_Orders
Filter: {'pickup_zone': 'Central', 'service_type': 'Parcel'}
Documents returned: 55
Documents examined: 55
Execution time ms: 1
Winning plan:
{'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'pickup_zone': 1, 'service_type': 1}, 'indexName': 'idx_zone_service', 'isMultiKey': False, 'multiKeyPaths': {'pickup_zone': [], 'service_type': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'pickup_zone': ['["Central", "Central"]'], 'service_type': ['["Parcel", "Parcel"]']}}}


## Step 21: Query optimisation interpretation

The query was tested using MongoDB's `explain` command. The purpose was to check whether MongoDB scanned the full collection or used an index.

The query filtered orders by `pickup_zone` and `service_type`. An index was created on both fields:

`idx_zone_service`

After indexing, the explain output should show that MongoDB can use an indexed query plan instead of scanning all documents. This supports query optimisation because it reduces the number of documents examined and improves retrieval efficiency for common operational queries.

# Final Summary

This notebook demonstrates MongoDB development for the NorthStar case study.

It includes:

- loading cleaned JSON files into MongoDB Atlas;
- designing separate operational collections;
- creating an integrated `Operational_Cases` document model;
- applying CRUD operations;
- using aggregation pipelines for analysis;
- joining related collections using `$lookup`;
- creating indexes;
- checking query performance using `explain`.

